In [ ]:
import os
import sys
import json
from pathlib import Path
from urllib.parse import urljoin
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Ensure `week1/scraper.py` is importable from this nested notebook folder.
for p in [Path.cwd(), *Path.cwd().parents]:
    scraper_dir = p / "week1"
    if (scraper_dir / "scraper.py").exists():
        sys.path.insert(0, str(scraper_dir))
        break

from scraper import fetch_website_contents, fetch_website_links

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key:
    print("API Key found and running")
else:
    print("API key not found")

MODEL = "gemini-2.5-flash-lite"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

In [ ]:
links=fetch_website_links("https://pro.spyn.co/")
links

In [ ]:
link_system_prompt = """
You are given a list of links from a competitor website.
Read and Select links that are most relevant as an activity/academy center management platform
the content from which can be a part of a new AI-native SaaS platform brochure that provides the same services .

Return ONLY a JSON object in this exact shape:
{
  "links": [
    {"type": "about page", "url": "https://example.com/class/about"},
    {"type": "careers", "url": "https://example.com/class/career"}
  ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    

    user_prompt = f"""
    I have shared a list of links from SpynPRO. 
    Decide the urls that will be helpful to make a company brochure for a competitor. 
    Respond with the full https URL in JSON format. 
    You can also quickly check the demo transcript, if possible to see what can be covered in the brochure
    Do not include Terms of Service, privacy policy pages, emails, release notes, and contact us pages.

Links (some might be relative links)
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://pro.spyn.co/"))

In [ ]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    normalized_links = []
    for link in links.get("links", []):
        if not isinstance(link, dict):
            continue
        absolute_url = urljoin(url, link.get("url", ""))
        if absolute_url.startswith(("http://", "https://")):
            normalized_links.append({**link, "url": absolute_url})

    links["links"] = normalized_links
    print(f"found {len(links['links'])} relevant links")
    return links


In [ ]:
select_relevant_links("https://pro.spyn.co/")

In [ ]:
def fetch_page_and_all_relevant_links(url):
    website_content = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page: \n\n{website_content}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        link_url = urljoin(url, link['url'])
        if not link_url.startswith(("http://", "https://")):
            continue
        result += f"\n\n## Link: {link['type']}\n"
        result += fetch_website_contents(link_url)
    return result


In [ ]:
print (fetch_page_and_all_relevant_links("https://pro.spyn.co/"))

In [ ]:
brochure_system_prompt = """ 
You are a marketing professional who is responsible to build a startup's brochure based on the competitors' links.
Analyze the content of the relevant links.
Respond in markdown without code blocks.
Include details about customers, competition and govenrment regulations.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are starting up a company: {company_name}.
    Here are the contents of its website and the relevant pages.
    Use this information to build a short brochure in markdown without code blocks. \n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [ ]:
get_brochure_user_prompt("SpynPRO", "https://pro.spyn.co/")

In [17]:
def create_company_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
       
    )
    result = response.choices[0].message.content
    display(Markdown(result)) 

In [18]:
create_company_brochure("SpynPRO", "https://pro.spyn.co/")

found 20 relevant links


# SpynPRO: Automate & Grow Your Sports & Fitness Business

SpynPRO is your all-in-one solution designed to streamline the operations of sports academies, fitness studios, gyms, and booking venues. We empower you to manage clients, payments, bookings, and events, allowing you to focus on what you do best – helping people achieve their fitness and sporting goals.

## Our Solutions

**AcademyPRO:** The ultimate tool for managing sports and hobby academies, fitness and dance studios, and gyms. Track client subscriptions, attendance, payments, and performance reports with ease.

**BookingsPRO:** For venues, this product enables seamless online and offline slot booking and package management, maximizing your venue's utilization.

**EventsPRO:** Simplify the creation, promotion, and management of ticketed events. Track check-ins, manage attendees, and keep your event stakeholders informed.

## Key Features to Elevate Your Business:

*   **Automated Payments & Reminders:** Get paid on time with automated payment reminders, online payment links, and integrated WhatsApp notifications.
*   **Client Management:** Maintain a comprehensive client database with detailed profiles, including transaction history, attendance, and performance.
*   **Attendance Tracking:** Effortlessly mark client attendance and manage makeup sessions. Auto-notify parents of absences.
*   **Performance Reporting:** Create custom assessment templates and allow coaches to easily grade students with remarks and images.
*   **Enquiry Management:** Streamline your lead generation with follow-up reminders and comment notifications to ensure high conversion rates.
*   **Staff Management:** Track staff attendance with geofencing capabilities for accurate check-ins and manage staff salaries.
*   **Engaging Communication:** Use in-built chat and messaging features to communicate with clients, share updates, collect feedback, and strengthen relationships.
*   **Referral Programs:** Encourage client referrals with built-in "Refer & Win" campaigns to fuel business growth.
*   **Branded App:** Offer your clients a personalized experience with your own branded app for viewing their transactions, attendance, and messages.
*   **Business Insights:** Gain valuable insights with a comprehensive business and expense dashboard.

## Our Customers

SpynPRO serves a diverse range of businesses including:

*   Sports Academies (football, cricket, martial arts, etc.)
*   Fitness Studios (yoga, Pilates, CrossFit, etc.)
*   Gyms and Fitness Centers
*   Dance Studios
*   Hobby Classes
*   Venue Owners managing sports facilities or activity spaces
*   Event Organizers

We cater to businesses of all sizes, from small local academies to larger fitness chains, operating in over 20 nations worldwide.

## Competition

The market for sports and fitness management software is competitive. Key competitors offer similar functionalities such as client management, scheduling, payment processing, and marketing tools. SpynPRO differentiates itself through its integrated suite of three distinct products (AcademyPRO, BookingsPRO, EventsPRO) addressing a broader spectrum of business needs within the sports and fitness ecosystem. Our focus on seamless WhatsApp integration, branded client apps, and robust performance reporting also provides a competitive edge.

## Government Regulations

While SpynPRO is a software solution, businesses using our platform must adhere to relevant government regulations concerning:

*   **Data Privacy:** Compliance with data protection laws such as GDPR (in Europe) or similar regulations in other regions is crucial when handling client personal information. SpynPRO is designed with data security in mind, but ultimate responsibility for compliance lies with the user.
*   **Payment Processing:** Businesses must comply with financial regulations, including those related to online transactions, payment gateway requirements, and consumer protection laws.
*   **Employment Law:** For staff management features, businesses need to ensure compliance with local labor laws regarding attendance, wages, and employee rights.
*   **Consumer Rights:** Laws pertaining to service provision, cancellations, and refunds for clients should be considered.

SpynPRO aims to provide tools that facilitate compliance, but it is the responsibility of the business owner to ensure their operations meet all legal requirements in their jurisdiction.